# Tutorial 2: Local Pools — Filesystem, Redis, HDF5

Store and recall a small random image-description dataset across three local backends using a single loop.

This demonstrates LAILA's uniform `memorize` / `remember` / `forget` interface — the code is identical regardless of the storage backend.

In [ ]:
import numpy as np
import laila
from laila.data import FilesystemPool, RedisPool, HDF5Pool

## Create the pools

Each pool manages its own storage automatically:
- **FilesystemPool** — local disk image
- **RedisPool** — private redis-server on a UNIX socket (no system Redis needed)
- **HDF5Pool** — local `.h5py` file

In [ ]:
pools = [
    ("fs",    FilesystemPool(nickname="tutorial_fs")),
    ("redis", RedisPool(nickname="tutorial_redis")),
    ("hdf5",  HDF5Pool(nickname="tutorial_hdf5")),
]

## Build a random image-description dataset

Each datapoint is a random RGB image plus a short caption. Both become constant entries, grouped under keys `image_0000` … `image_0004`.

In [ ]:
CAPTIONS = [
    "a red bicycle leaning against a brick wall",
    "two cats sleeping on a sunny windowsill",
    "a lighthouse on a cliff at dusk",
    "a bowl of ramen with a soft-boiled egg",
    "a snow-covered pine forest at dawn",
]
N_IMAGES = len(CAPTIONS)

dataset = {}
for i in range(N_IMAGES):
    image = np.random.randint(0, 256, size=(64, 64, 3), dtype=np.uint8)
    dataset[f"image_{i:04d}"] = {
        "image": laila.constant(data=image),
        "text": laila.constant(data=CAPTIONS[i]),
    }

print(f"{N_IMAGES} image/caption pairs, {2 * N_IMAGES} entries")
print("image_0000 image ->", dataset["image_0000"]["image"].global_id)
print("image_0000 text  ->", dataset["image_0000"]["text"].global_id)

## Register, memorize, remember, verify — in a loop

For each backend we memorize every entry and keep **only its `global_id`** in a `blueprint` dict that mirrors the dataset structure. The blueprint is all we need to remember everything back and clean up.

The key point: the code inside the loop is identical for all three backends.

In [ ]:
for nick, pool in pools:
    laila.memory.extend(pool, pool_nickname=nick)

    # Memorize every entry, keeping only its global_id in the blueprint
    blueprint = {}
    with laila.guarantee:
        for i in range(N_IMAGES):
            blueprint[f"image_{i:04d}"] = {}

            entry = dataset[f"image_{i:04d}"]["image"]
            laila.memorize(entry, dst_pool=nick)
            blueprint[f"image_{i:04d}"]["image"] = entry.global_id  # keep only the id

            entry = dataset[f"image_{i:04d}"]["text"]
            laila.memorize(entry, dst_pool=nick)
            blueprint[f"image_{i:04d}"]["text"] = entry.global_id  # keep only the id
    print(f"[{nick}] memorized {2 * N_IMAGES} entries")

    # Remember everything back from the ids alone
    with laila.guarantee:
        recalled = {
            key: {field: laila.remember(gid, dst_pool=nick) for field, gid in ids.items()}
            for key, ids in blueprint.items()
        }
    print(f"[{nick}] remembered — image_0000: {recalled['image_0000']['image'].data.shape}, {recalled['image_0000']['text'].data!r}")

    # Verify against the original dataset
    for key in blueprint:
        assert np.array_equal(dataset[key]["image"].data, recalled[key]["image"].data), f"Image mismatch on {nick}/{key}!"
        assert dataset[key]["text"].data == recalled[key]["text"].data, f"Text mismatch on {nick}/{key}!"
    print(f"[{nick}] verified")

    # Clean up
    with laila.guarantee:
        for ids in blueprint.values():
            for gid in ids.values():
                laila.forget(gid, pool=nick)
    print(f"[{nick}] cleaned up\n")

## What just happened

1. **`laila.memory.extend`** registered each backend under a nickname.
2. **`laila.memorize`** serialized each entry (a NumPy image or a caption string), applied the pool's transformations, and wrote to the backend.
3. **`with laila.guarantee:`** blocked until every future created inside the block had completed — ten memorizes per pool, no manual `wait()` calls needed.
4. The **`blueprint`** kept only `global_id` strings, mirroring the dataset's `image_XXXX / {image, text}` structure. That is exactly what a `Manifest` builds for you automatically — see Tutorial 5.
5. **`laila.remember`** fetched each blob by id, reversed the transformations, and reconstructed the entry. The returned future exposes the recovered payload via `.data`.
6. **`laila.forget`** deleted every stored blob.

### Addressing a pool

`memorize` / `remember` select the pool with `dst_pool=` and `forget` with `pool=`. Each accepts a registered **nickname** (as above), a pool **`global_id`**, or a live **pool object**. When you omit it, the call goes to the alpha (default, in-memory) pool. The older `pool_nickname=` / `pool_id=` keywords are still accepted as back-compat aliases.

Every backend LAILA supports — S3, GCS, Azure, Postgres, MongoDB, and more — answers to the same three verbs. Only the pool constructor changes.

**Next:** Tutorial 3 — S3 with NumPy and PyTorch Tensors